# XGBoost + Optuna: solamente mejores hiperparámetros

El notebook termina inmediatamente después de obtener `best_params`.

No entrena el modelo final, no genera predicciones OOF finales, no crea gráficas y no ejecuta SHAP ni clustering.


## 1. Instalar dependencias


In [ ]:
# Ejecuta esta celda en Colab o Jupyter si no tienes las librerías.
%pip install -q optuna xgboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.2 MB/s eta 0:00:00


## 2. Imports y configuración


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

import optuna
import xgboost as xgb


# ------------------------------------------------------------
# EDITA ESTAS OPCIONES
# ------------------------------------------------------------
DATA_PATH = Path("/content/ENSANUT_2024_mx.csv")
OUTPUT_DIR = Path("/content/resultados_optuna")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "HB1AC"

N_SPLITS = 5
N_TRIALS = 50#35
RANDOM_STATE = 42

# Usa "cpu" si no tienes GPU compatible.
XGB_DEVICE = "cuda"

print("XGBoost:", xgb.__version__)
print("Optuna:", optuna.__version__)


XGBoost: 3.3.0
Optuna: 4.9.0


## 3. Cargar la base


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró la base: {DATA_PATH}. "
        "Corrige DATA_PATH o sube el archivo."
    )

df_model = pd.read_csv(DATA_PATH, low_memory=False)

if TARGET_COLUMN not in df_model.columns:
    raise KeyError(f"No existe la variable target: {TARGET_COLUMN}")

print(f"Base cargada: {df_model.shape[0]:,} filas × {df_model.shape[1]:,} columnas")


Base cargada: 36,021 filas × 2,697 columnas


## 4. Preparar `X` y `y`


In [ ]:
ID_COLUMNS = [
    "FOLIO_I",
    "FOLIO_INT",
]

TARGET_DERIVED_COLUMNS = [
    "clasif_hba1c_cat",
    "clasif_hba1c_bin",
    "clasif_combinada_pref_bin",
    "clasif_combinada_pref_cat",
]

OTHER_EXCLUDE = [
    "gpoedad",
    "annac",
    "mesnac",
    "fecha_nacimiento",
    "H0304A",
    "H0304M",
    "FECH_NAC",
    "edad",
    "AEDAD",
    "A0302",
]

MANUAL_EXCLUDE = []

y = pd.to_numeric(df_model[TARGET_COLUMN], errors="coerce")

cols_drop = (
    ID_COLUMNS
    + TARGET_DERIVED_COLUMNS
    + OTHER_EXCLUDE
    + MANUAL_EXCLUDE
    + [TARGET_COLUMN]
)

# Corrección: debe ser df_model.drop(...), no model.drop(...).
X = df_model.drop(
    columns=[c for c in cols_drop if c in df_model.columns],
    errors="ignore",
).copy()

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

X[num_cols] = X[num_cols].apply(pd.to_numeric, errors="coerce")

# Se conserva tu decisión de no imputar las variables numéricas.
# X[num_cols] = X[num_cols].fillna(X[num_cols].median())

X[cat_cols] = X[cat_cols].astype("string").fillna("MISSING")
X = pd.get_dummies(
    X,
    columns=cat_cols,
    drop_first=False,
)

ok = y.notna()
X = X.loc[ok].reset_index(drop=True)
y = y.loc[ok].reset_index(drop=True)

X_values = X.astype(np.float32).values
y_values = y.astype(np.float32).values

print(f"Matriz final: {X_values.shape[0]:,} filas × {X_values.shape[1]:,} predictores")


Matriz final: 2,530 filas × 20,154 predictores


## 5. Optuna con 5-fold CV


In [ ]:
cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Los mismos folds se reutilizan en todos los ensayos.
cv_splits = list(cv.split(X_values))


def crear_modelo(params):
    argumentos = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        **params,
    }

    # XGBoost 2.x y posteriores.
    if int(xgb.__version__.split(".")[0]) >= 2:
        argumentos["device"] = XGB_DEVICE
    elif XGB_DEVICE == "cuda":
        argumentos["tree_method"] = "gpu_hist"

    return xgb.XGBRegressor(**argumentos)


def evaluar_cv(params, trial=None):
    r2_folds = []

    for fold, (tr, va) in enumerate(cv_splits, start=1):
        model = crear_modelo(params)

        model.fit(
            X_values[tr],
            y_values[tr],
            verbose=False,
        )

        pred = model.predict(X_values[va])
        r2_fold = r2_score(y_values[va], pred)
        r2_folds.append(r2_fold)

        if trial is not None:
            trial.report(
                float(np.mean(r2_folds)),
                step=fold,
            )

            if trial.should_prune():
                raise optuna.TrialPruned()

    return float(np.mean(r2_folds))


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            900,
            2000,
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.08,
            log=True,
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            10,
            15,
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10,
        ),
        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0,
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0,
        ),
        "gamma": trial.suggest_float(
            "gamma",
            0.0,
            5.0,
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-4,
            10.0,
            log=True,
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            20.0,
            log=True,
        ),
    }

    return evaluar_cv(params, trial=trial)


study = optuna.create_study(
    direction="maximize",
    study_name="xgb_hba1c_reg",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE,
    ),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10,
        n_warmup_steps=2,
    ),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_params = study.best_params

print("\nMEJORES HIPERPARÁMETROS:")
print(json.dumps(best_params, indent=2, ensure_ascii=False))

print("\nMEJOR R² PROMEDIO DE LOS FOLDS:")
print(round(study.best_value, 6))

with open(
    OUTPUT_DIR / "mejores_hiperparametros_optuna.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        best_params,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    "\nArchivo guardado en:",
    OUTPUT_DIR / "mejores_hiperparametros_optuna.json",
)


[I 2026-07-15 02:18:29,361] A new study created in memory with name: xgb_hba1c_reg


  0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:18:31] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:18:31] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:18:57] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:18:57] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[I 2026-07-15 02:20:36,621] Trial 0 finished with value: 0.7893991470336914 and parameters: {'n_estimators': 1312, 'learning_rate': 0.07220721098226064, 'max_depth': 14, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 2.1423021757741068, 'reg_lambda': 0.3849583075868115}. Best is trial 0 with value: 0.7893991470336914.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:20:39] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:20:39] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:21:20] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:21:20] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


...

[I 2026-07-15 04:38:49,264] Trial 47 finished with value: 0.7968920588493347 and parameters: {'n_estimators': 1160, 'learning_rate': 0.017722614498396627, 'max_depth': 14, 'min_child_weight': 7, 'subsample': 0.6706521295311052, 'colsample_bytree': 0.9740601444136457, 'gamma': 1.6364463669475187, 'reg_alpha': 0.00010804240668636775, 'reg_lambda': 0.0017520552554048453}. Best is trial 46 with value: 0.8021199226379394.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [04:38:51] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [04:38:51] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [04:39:24] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [04:39:24] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[I 2026-07-15 04:41:28,771] Trial 48 finished with value: 0.8030649423599243 and parameters: {'n_estimators': 1339, 'learning_rate': 0.01109231653987982, 'max_depth': 15, 'min_child_weight': 10, 'subsample': 0.604285125906042, 'colsample_bytree': 0.9741262828257075, 'gamma': 2.0067107351969584, 'reg_alpha': 0.0032372287050650432, 'reg_lambda': 0.00943910513138246}. Best is trial 48 with value: 0.8030649423599243.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [04:41:30] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [04:41:30] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [04:42:02] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [04:42:02] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[I 2026-07-15 04:44:10,190] Trial 49 finished with value: 0.8021614074707031 and parameters: {'n_estimators': 1353, 'learning_rate': 0.011022382104940475, 'max_depth': 15, 'min_child_weight': 10, 'subsample': 0.6466381742393899, 'colsample_bytree': 0.998747807259214, 'gamma': 1.9401151089109916, 'reg_alpha': 0.0014044434427136687, 'reg_lambda': 0.004761120232963683}. Best is trial 48 with value: 0.8030649423599243.

MEJORES HIPERPARÁMETROS:
{
  "n_estimators": 1339,
  "learning_rate": 0.01109231653987982,
  "max_depth": 15,
  "min_child_weight": 10,
  "subsample": 0.604285125906042,
  "colsample_bytree": 0.9741262828257075,
  "gamma": 2.0067107351969584,
  "reg_alpha": 0.0032372287050650432,
  "reg_lambda": 0.00943910513138246
}

MEJOR R² PROMEDIO DE LOS FOLDS:
0.803065

Archivo guardado en: /content/resultados_optuna/mejores_hiperparametros_optuna.json


## Fin

Los mejores hiperparámetros están en:

```python
best_params
```

El notebook termina aquí.


In [ ]:
MEJORES HIPERPARÁMETROS:
{
  "n_estimators": 1898,
  "learning_rate": 0.037884045523275636,
  "max_depth": 13,
  "min_child_weight": 10,
  "subsample": 0.7671330257521887,
  "colsample_bytree": 0.9086499720541381,
  "gamma": 2.072958782974687,
  "reg_alpha": 0.01472916636028212,
  "reg_lambda": 0.0028944665294875532
}

MEJOR R² PROMEDIO DE LOS FOLDS:
0.800013


SyntaxError: invalid character '²' (U+00B2) (170422963.py, line 14)

In [ ]:
MEJORES HIPERPARÁMETROS:
{
  "n_estimators": 1339,
  "learning_rate": 0.01109231653987982,
  "max_depth": 15,
  "min_child_weight": 10,
  "subsample": 0.604285125906042,
  "colsample_bytree": 0.9741262828257075,
  "gamma": 2.0067107351969584,
  "reg_alpha": 0.0032372287050650432,
  "reg_lambda": 0.00943910513138246
}

MEJOR R² PROMEDIO DE LOS FOLDS:
0.803065